## A character-level implementation of [Bengio et al. 2003](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)

In their original paper they associate each of 17,000 words with a 30-dimensional vector (called feature vector).

This is a concept of (vector) embedding of words.

这些 associations 被存放在了一个 lookup table 里：一个 17000 * 30 的矩阵。

In the beginning, all the points (i.e., the feature vectors of the words) in this 30-dimensional space are initialized randomly, so they are spread out into all directions in this space. Then using backpropagation those feature vectors are tuned So that words with similar meaning move close together, and words that are semantically different are pointing in different directions in this (semantic) space.

They <span style="color:red">maximize the log likelihood</span> to train the multi-layer neural network.

During test time, prefix could be a sequence that never appears in the training set. 但 prefix 中的很多 tokens 在 semantic space 里和 neural network 见过的很多 tokens 离得很近。<span style="color:red">Neural network 大概知道这些它见过的 tokens 后面的 prediction 是什么</span>，所以 neural network 就能对这个没见过的 sequence 给出一个相似的 prediction。

In [7]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [8]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [9]:
len(words)

32033

### Vocabulary & Mappings to/from indices

In [14]:
letters = sorted(list(set(''.join(words))))
str_to_idx = {s:i+1 for i, s in enumerate(letters)}
str_to_idx['.'] = 0
idx_to_str = {i:s for s, i in str_to_idx.items()}

print(idx_to_str)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


### Build the Dataset

In [21]:
block_size = 3  # size of the prefix context
X = []  # inputs
Y = []  # labels

for w in words[:2]:
    print(w)
    context = [0] * block_size  # [0,0,...,0]
    # Note here that we use the special token '.' for padding
    for ch in w + '.':
        index = str_to_idx[ch]
        X.append(context)
        Y.append(index)

        print(''.join(idx_to_str[i] for i in context), '--->', idx_to_str[index])

        context = context[1:] + [index]   # roll the window

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .


In [22]:
X.shape, X.dtype

(torch.Size([12, 3]), torch.int64)

In [23]:
Y.shape, Y.dtype

(torch.Size([12]), torch.int64)

原论文有17,000个词汇，而我们有27个词汇（26 个英文字母 + 一个 special token）。

原论文用了30个维度去 embed 那17,000个词汇。

<span style="color:red">我们只需要用2个维度去 embed 这27个词汇。</span>

In [24]:
M = torch.randn((27, 2))
M


tensor([[ 1.2126, -0.7901],
        [ 0.1946,  0.1449],
        [ 0.0052, -2.0909],
        [-0.5683, -1.1494],
        [-0.5149, -1.1429],
        [-1.6300, -0.6034],
        [-0.8087, -0.0160],
        [ 0.2961, -0.9621],
        [-0.5701,  0.6873],
        [ 2.0046, -0.0759],
        [-1.2360,  0.0998],
        [-0.4619,  0.4920],
        [-0.6483,  0.3150],
        [-1.2560,  0.8743],
        [-0.3782, -0.6839],
        [ 0.0274,  1.5557],
        [-0.4367,  0.7523],
        [ 1.1159,  1.3114],
        [-1.2829, -0.5588],
        [ 0.4740,  0.0243],
        [ 1.3330,  0.5576],
        [ 0.5046, -0.8349],
        [ 0.2790,  1.6121],
        [ 0.2104,  0.3096],
        [-0.7143,  1.1637],
        [ 1.5249, -0.8935],
        [-0.3893,  0.2398]])

我们可以用 one-hot vector 去提取某个词的 vector embedding:

In [27]:
M.dtype

torch.float32

In [29]:
F.one_hot(torch.tensor(5), num_classes=27).dtype

torch.int64

In [26]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ M

tensor([-1.6300, -0.6034])

这里我们可以把“从embedding matrix M中提取token对应的embedded vector”理解成“用one hot encoding去表示输入tokens，然后把这些encoding（拼成一个input）feed进一个bias为 0 的linear layer里（这个layer中有embedding dimension * block_size个神经元，每个神经元有 vocabulary size * block_size 个weights），这个 linear layer 的输出就是这些 tokens 的 embedded vectors”